In [ ]:
from experta import *

class State(Fact):
    pass

class BridgeExpertSystem(KnowledgeEngine):
    def __init__(self):
        super().__init__()
        self.persons_time = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}
        self.max_time = 17
        self.visited = set()
        self.tree = {}
        self.node_counter = 0

    @DefFacts()
    def _initial_state(self):
        print("\n🚀 بدأ البحث...\n")
        yield State(left=('me', 'lab', 'worker', 'scientist'), right=(), light='left', time=0, path=(), node=0, parent=None)

    #  هذه القاعدة للسماح لشخص واحد بالعبور إذا كان وحده في اليسار
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node),
        TEST(lambda left: len(left) == 1))
    def single_cross_to_right(self, left, right, time, path, node):
        p = left[0]
        new_left = []
        new_right = list(right) + [p]
        duration = self.persons_time[p]
        new_time = time + duration
        if new_time <= self.max_time:
            step = f"{p} crossed alone to right in {duration} min"
            self.generate_state_from_lists(new_left, new_right, 'right', new_time, path, step, node)

    # الانتقال من اليسار إلى اليمين (شخصين)
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path, node=MATCH.node))
    def cross_to_right(self, left, right, time, path, node):
        left_list = list(left)
        for i in range(len(left_list)):
            for j in range(i + 1, len(left_list)):
                p1 = left_list[i]
                p2 = left_list[j]
                new_left = left_list[:]
                new_right = list(right)
                new_left.remove(p1)
                new_left.remove(p2)
                new_right += [p1, p2]
                duration = max(self.persons_time[p1], self.persons_time[p2])
                new_time = time + duration
                if new_time <= self.max_time:
                    step = f"{p1} and {p2} crossed to right in {duration} min"
                    self.generate_state_from_lists(new_left, new_right, 'right', new_time, path, step, node)




    # الرجوع من اليمين إلى اليسار (شخص واحد)
    @Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path, node=MATCH.node))
    def return_to_left(self, left, right, time, path, node):
        for p in right:
            new_left = list(left) + [p]
            new_right = list(right)
            new_right.remove(p)
            duration = self.persons_time[p]
            new_time = time + duration
            if new_time <= self.max_time:
                step = f"{p} returned to left in {duration} min"
                self.generate_state_from_lists(new_left, new_right, 'left', new_time, path, step, node)

    @Rule(State(time=MATCH.time),
          TEST(lambda time: time > 17),
          salience=900)
    def remove_overtime_state(self, time):
        print(f"⛔️ حالة تم تجاوز الوقت ({time} دقيقة). تجاهلها.")
        #self.halt()
        pass

    def generate_state_from_lists(self, left_list, right_list, light, time, path, step, parent):
        print(f"Generated new state: Left={left_list}, Right={right_list}, Light={light}, Time={time}, Step={step}")

        signature = (tuple(sorted(left_list)), tuple(sorted(right_list)), light, time)
        if signature in self.visited:
            print("⚠️ تم تجاهل حالة مكررة.")
            return
        
        self.visited.add(signature)
        self.node_counter += 1
        node = self.node_counter
        self.tree[node] = {'parent': parent, 'left': tuple(left_list), 'right': tuple(right_list), 'light': light, 'time': time}
        new_path = list(path) + [step]
        #print(f"🔄 حالة جديدة | وقت: {time} | ضوء: {light} | يسار: {left_list} | يمين: {right_list} | خطوة: {step}")
        self.declare(State(left=tuple(left_list), right=tuple(right_list), light=light, time=time, path=tuple(new_path), node=node, parent=parent))

    @Rule(State(left=MATCH.left, right=MATCH.right, time=MATCH.time, path=MATCH.path),
          TEST(lambda left, right: set(left) == set() and set(right) == {'me', 'lab', 'worker', 'scientist'}),
          salience=1000)
    def goal_reached(self, left, right, time, path):
        print("\n✅✅ تم الوصول إلى الهدف في:", time, "دقيقة")
        print("\n📜 خطوات الحل:")
        for i, step in enumerate(path, 1):
            print(f" {i}. {step}")
        print("\n🌳 شجرة البحث:")
        for node_id, data in self.tree.items():
            print(f"🔸 Node {node_id} (Parent: {data['parent']}) | Left: {data['left']} | Right: {data['right']} | Light: {data['light']} | Time: {data['time']}")
        self.halt()

# --- تشغيل النظام ---
engine = BridgeExpertSystem()
engine.reset()
engine.run()



🚀 بدأ البحث...

Generated new state: Left=['worker', 'scientist'], Right=['me', 'lab'], Light=right, Time=2, Step=me and lab crossed to right in 2 min
Generated new state: Left=['lab', 'scientist'], Right=['me', 'worker'], Light=right, Time=5, Step=me and worker crossed to right in 5 min
Generated new state: Left=['lab', 'worker'], Right=['me', 'scientist'], Light=right, Time=10, Step=me and scientist crossed to right in 10 min
Generated new state: Left=['me', 'scientist'], Right=['lab', 'worker'], Light=right, Time=5, Step=lab and worker crossed to right in 5 min
Generated new state: Left=['me', 'worker'], Right=['lab', 'scientist'], Light=right, Time=10, Step=lab and scientist crossed to right in 10 min
Generated new state: Left=['me', 'lab'], Right=['worker', 'scientist'], Light=right, Time=10, Step=worker and scientist crossed to right in 10 min
Generated new state: Left=['me', 'lab', 'worker'], Right=['scientist'], Light=left, Time=15, Step=worker returned to left in 5 min
Genera

In [1]:
from experta import *

class State(Fact):
    salience = Field(int, default=900)

class BridgeExpertSystem(KnowledgeEngine):
    def __init__(self, mode='dfs'):
        super().__init__()
        self.persons_time = {'me': 1, 'lab': 2, 'worker': 5, 'scientist': 10}
        self.max_time = 17
        self.visited = set()
        self.tree = {}
        self.node_counter = 0
        self.mode = mode

    @DefFacts()
    def _initial_state(self):
        print(f"\n🚀 بدأ البحث بوضع {self.mode.upper()}...\n")
        yield State(left=('me', 'lab', 'worker', 'scientist'), right=(), light='left',
                    time=0, path=(), node=0, parent=None, depth=0, salience=900)

    # ========== قواعد DFS ==========
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path,
                node=MATCH.node, depth=MATCH.depth),
          TEST(lambda: True))
    def single_cross_dfs(self, left, right, time, path, node, depth):
        if self.mode != 'dfs' or len(left) != 1:
            return
        self.cross_single(left, right, time, path, node, depth)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path,
                node=MATCH.node, depth=MATCH.depth),
          TEST(lambda: True))
    def cross_to_right_dfs(self, left, right, time, path, node, depth):
        if self.mode != 'dfs':
            return
        self.cross_pairs(left, right, time, path, node, depth)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path,
                node=MATCH.node, depth=MATCH.depth),
          TEST(lambda: True))
    def return_to_left_dfs(self, left, right, time, path, node, depth):
        if self.mode != 'dfs':
            return
        self.return_back(left, right, time, path, node, depth)

    # ========== قواعد BFS ==========
    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path,
                node=MATCH.node, depth=MATCH.depth),
          TEST(lambda: True))
    def single_cross_bfs(self, left, right, time, path, node, depth):
        if self.mode != 'bfs' or len(left) != 1:
            return
        self.cross_single(left, right, time, path, node, depth)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='left', time=MATCH.time, path=MATCH.path,
                node=MATCH.node, depth=MATCH.depth),
          TEST(lambda: True))
    def cross_to_right_bfs(self, left, right, time, path, node, depth):
        if self.mode != 'bfs':
            return
        self.cross_pairs(left, right, time, path, node, depth)

    @Rule(State(left=MATCH.left, right=MATCH.right, light='right', time=MATCH.time, path=MATCH.path,
                node=MATCH.node, depth=MATCH.depth),
          TEST(lambda: True))
    def return_to_left_bfs(self, left, right, time, path, node, depth):
        if self.mode != 'bfs':
            return
        self.return_back(left, right, time, path, node, depth)

    # ========== دوال مشتركة ==========
    def cross_single(self, left, right, time, path, node, depth):
        p = left[0]
        new_left = []
        new_right = list(right) + [p]
        duration = self.persons_time[p]
        new_time = time + duration
        if new_time <= self.max_time:
            step = f"{p} crossed alone to right in {duration} min"
            self.generate_state_from_lists(new_left, new_right, 'right', new_time, path, step, node, depth + 1)

    def cross_pairs(self, left, right, time, path, node, depth):
        for i in range(len(left)):
            for j in range(i + 1, len(left)):
                p1, p2 = left[i], left[j]
                new_left = list(left)
                new_left.remove(p1)
                new_left.remove(p2)
                new_right = list(right) + [p1, p2]
                duration = max(self.persons_time[p1], self.persons_time[p2])
                new_time = time + duration
                if new_time <= self.max_time:
                    step = f"{p1} and {p2} crossed to right in {duration} min"
                    self.generate_state_from_lists(new_left, new_right, 'right', new_time, path, step, node, depth + 1)

    def return_back(self, left, right, time, path, node, depth):
        for p in right:
            new_left = list(left) + [p]
            new_right = list(right)
            new_right.remove(p)
            duration = self.persons_time[p]
            new_time = time + duration
            if new_time <= self.max_time:
                step = f"{p} returned to left in {duration} min"
                self.generate_state_from_lists(new_left, new_right, 'left', new_time, path, step, node, depth + 1)

    def generate_state_from_lists(self, left_list, right_list, light, time, path, step, parent, depth):
        salience = 900 - depth if self.mode == 'dfs' else 900 + depth
        signature = (tuple(sorted(left_list)), tuple(sorted(right_list)), light, time)
        if signature in self.visited:
            return

        print(f"🎯 حالة جديدة: Left={left_list}, Right={right_list}, Light={light}, Time={time}, Step={step}, Depth={depth}")
        self.visited.add(signature)
        self.node_counter += 1
        node = self.node_counter
        self.tree[node] = {
            'parent': parent, 'left': tuple(left_list), 'right': tuple(right_list),
            'light': light, 'time': time, 'depth': depth
        }

        new_path = list(path) + [step]
        self.declare(State(left=tuple(left_list), right=tuple(right_list), light=light,
                           time=time, path=tuple(new_path), node=node, parent=parent,
                           depth=depth, salience=salience))

    # ========== الهدف ==========
    @Rule(State(left=MATCH.left, right=MATCH.right, time=MATCH.time, path=MATCH.path),
          TEST(lambda left, right: set(left) == set() and set(right) == {'me', 'lab', 'worker', 'scientist'}),
          salience=1000)
    def goal_reached(self, left, right, time, path):
        print("\n✅ تم الوصول إلى الهدف خلال:", time, "دقيقة")
        print("\n📜 خطوات الحل:")
        for i, step in enumerate(path, 1):
            print(f"  {i}. {step}")
        print("\n🌳 شجرة البحث:")
        for node_id, data in self.tree.items():
            print(f"🔸 Node {node_id} (Parent: {data['parent']}, Depth: {data['depth']}) "
                  f"| Left: {data['left']} | Right: {data['right']} | Light: {data['light']} | Time: {data['time']}")
        self.halt()

        # --- البحث بـ DFS ---
print("\n===== البحث باستخدام DFS =====")
engine_dfs = BridgeExpertSystem(mode='dfs')
engine_dfs.reset()
engine_dfs.run()

# --- البحث بـ BFS ---
print("\n===== البحث باستخدام BFS =====")
engine_bfs = BridgeExpertSystem(mode='bfs')
engine_bfs.reset()
engine_bfs.run()




===== البحث باستخدام DFS =====

🚀 بدأ البحث بوضع DFS...

🎯 حالة جديدة: Left=['worker', 'scientist'], Right=['me', 'lab'], Light=right, Time=2, Step=me and lab crossed to right in 2 min, Depth=1
🎯 حالة جديدة: Left=['lab', 'scientist'], Right=['me', 'worker'], Light=right, Time=5, Step=me and worker crossed to right in 5 min, Depth=1
🎯 حالة جديدة: Left=['lab', 'worker'], Right=['me', 'scientist'], Light=right, Time=10, Step=me and scientist crossed to right in 10 min, Depth=1
🎯 حالة جديدة: Left=['me', 'scientist'], Right=['lab', 'worker'], Light=right, Time=5, Step=lab and worker crossed to right in 5 min, Depth=1
🎯 حالة جديدة: Left=['me', 'worker'], Right=['lab', 'scientist'], Light=right, Time=10, Step=lab and scientist crossed to right in 10 min, Depth=1
🎯 حالة جديدة: Left=['me', 'lab'], Right=['worker', 'scientist'], Light=right, Time=10, Step=worker and scientist crossed to right in 10 min, Depth=1
🎯 حالة جديدة: Left=['me', 'lab', 'worker'], Right=['scientist'], Light=left, Time=15